# HW02：列表、字典和类

学号：20244080115　姓名：胡勇彦

In [1]:
import hashlib
import os
import platform
import uuid

STUDENT_ID = 20244080115

CONDA_ENV = os.environ.get("CONDA_DEFAULT_ENV", "base")
print(f"[Conda 环境] {CONDA_ENV}")

MACHINE_FP = hashlib.sha256(
    f"{platform.node()}|{platform.machine()}|{platform.system()}|{uuid.getnode()}".encode()
).hexdigest()[:16]
RUN_ID = f"{STUDENT_ID}-{MACHINE_FP}"
print(f"[学号] {STUDENT_ID}")
print(f"[机器指纹] {MACHINE_FP}")
print(f"[RUN_ID] {RUN_ID}")

[Conda 环境] base
[学号] 20244080115
[机器指纹] c27f93f8bd6c2a0d
[RUN_ID] 20244080115-c27f93f8bd6c2a0d


## 一、字符串的乐趣

### 1. `is_palindrome`：判断回文，忽略空格和大小写

In [2]:
def is_palindrome(s):
    t = s.replace(" ", "").lower()
    return t == t[::-1]

for s in ["level", "kayak", "pop", "rats live on no evil star",
          "Was it a car or a cat I saw", "tacocat", "TACO cat", "hello"]:
    print(repr(s), "->", is_palindrome(s))

'level' -> True
'kayak' -> True
'pop' -> True
'rats live on no evil star' -> True
'Was it a car or a cat I saw' -> True
'tacocat' -> True
'TACO cat' -> True
'hello' -> False


### 2. `is_abecedarian`：字母是否按顺序排列，忽略空格

In [3]:
def is_abecedarian(s):
    t = s.replace(" ", "").lower()
    return all(t[i] <= t[i + 1] for i in range(len(t) - 1))

for s in ["adder", "beet", "dog", "cat", "abcd efgh xyz"]:
    print(repr(s), "->", is_abecedarian(s))

'adder' -> True
'beet' -> True
'dog' -> False
'cat' -> False
'abcd efgh xyz' -> True


### 3. `remove_vowels`：去掉所有元音 a/e/i/o/u

In [4]:
def remove_vowels(s):
    return "".join(ch for ch in s if ch.lower() not in "aeiou")

print(remove_vowels("cat"))
print(remove_vowels("audio"))
print(remove_vowels("aeiou"))

ct
d



## 二、列表的乐趣

### 1. `list_reverse`：返回反转后的列表；非列表输入引发错误

In [5]:
def list_reverse(lst):
    if not isinstance(lst, list):
        raise TypeError("input must be a list")
    return lst[::-1]

print(list_reverse([1, 2, 3]))
try:
    list_reverse((1, 2, 3))
except TypeError as e:
    print("TypeError:", e)

[3, 2, 1]
TypeError: input must be a list


### 2. `is_sorted`：单次遍历判断序列是否升序

In [6]:
def is_sorted(seq):
    for i in range(len(seq) - 1):
        if seq[i] > seq[i + 1]:
            return False
    return True

print(is_sorted([1, 2, 3, 4]))
print(is_sorted([1, 3, 2]))
print(is_sorted(("a", "b", "c")))

True
False
True


## 三、作为向量的元组

### 1. `vec_scalar_mult`：向量标量乘法，含类型检查

In [7]:
def vec_scalar_mult(t, s):
    if not isinstance(t, tuple) or not all(
        isinstance(x, (int, float)) and not isinstance(x, bool) for x in t
    ):
        raise TypeError("t must be a tuple of ints/floats")
    if not isinstance(s, (int, float)) or isinstance(s, bool):
        raise TypeError("s must be an int or float")
    return tuple(x * s for x in t)

print(vec_scalar_mult((1, 2, 3), 2))
print(vec_scalar_mult((1.5, 2), 0.5))
try:
    vec_scalar_mult([1, 2], 2)
except TypeError as e:
    print("TypeError:", e)

(2, 4, 6)
(0.75, 1.0)
TypeError: t must be a tuple of ints/floats


### 2. `vec_inner_product`：内积，检查类型与长度

In [8]:
def vec_inner_product(x, y):
    if not isinstance(x, tuple) or not isinstance(y, tuple):
        raise TypeError("both inputs must be tuples")
    if len(x) != len(y):
        raise ValueError("tuples must have the same length")
    return float(sum(a * b for a, b in zip(x, y)))

print(vec_inner_product((1, 2, 3), (4, 5, 6)))
try:
    vec_inner_product((1, 2), (3, 4, 5))
except ValueError as e:
    print("ValueError:", e)

32.0
ValueError: tuples must have the same length


### 3. `check_valid_mx`：检查“元组的元组”是否为有效矩阵

In [9]:
def check_valid_mx(mx):
    if not isinstance(mx, tuple) or len(mx) == 0:
        return False
    if not all(isinstance(row, tuple) for row in mx):
        return False
    n = len(mx[0])
    if any(len(row) != n for row in mx):
        return False
    return all(
        isinstance(v, (int, float)) and not isinstance(v, bool)
        for row in mx for v in row
    )

print(check_valid_mx(((1, 2), (3, 4))))
print(check_valid_mx(((1, 2), (3, 4, 5))))
print(check_valid_mx(([1, 2], [3, 4])))

True
False
False


### 4. `mx_vec_mult`：矩阵乘向量

In [10]:
def mx_vec_mult(mx, v):
    if not check_valid_mx(mx):
        raise TypeError("invalid matrix")
    if not isinstance(v, tuple) or not all(
        isinstance(x, (int, float)) and not isinstance(x, bool) for x in v
    ):
        raise TypeError("invalid vector")
    if len(mx[0]) != len(v):
        raise ValueError("dimension mismatch")
    return tuple(vec_inner_product(row, v) for row in mx)

mx = ((1, 2), (3, 4), (5, 6))
print(mx_vec_mult(mx, (7, 8)))

(23.0, 53.0, 83.0)


## 四、更多向量的乐趣：`Vector` 类

### 1 & 3-5. 类定义（初始化校验、比较运算符、`dot`、乘法运算符）

In [11]:
class Vector:
    def __init__(self, d, elements=None):
        if isinstance(d, bool) or not isinstance(d, int):
            raise TypeError("dimension must be an int")
        if d < 0:
            raise ValueError("dimension must be non-negative")
        if elements is None:
            elements = [0.0] * d          # 只给维度时默认零向量
        if not isinstance(elements, (list, tuple)) or not all(
            isinstance(x, (int, float)) and not isinstance(x, bool) for x in elements
        ):
            raise TypeError("elements must be a list/tuple of ints/floats")
        if len(elements) != d:
            raise ValueError("number of elements must match dimension")
        self.dimension = d
        self.elements = tuple(elements)

    def __eq__(self, other):
        if not isinstance(other, Vector):
            return NotImplemented
        if self.dimension != other.dimension:
            raise ValueError("cannot compare vectors of different dimensions")
        return self.elements == other.elements

    def __lt__(self, other):
        if not isinstance(other, Vector):
            return NotImplemented
        if self.dimension != other.dimension:
            raise ValueError("cannot compare vectors of different dimensions")
        return self.elements < other.elements

    def __le__(self, other):
        return self == other or self < other

    def __gt__(self, other):
        if not isinstance(other, Vector):
            return NotImplemented
        return not (self <= other)

    def __ge__(self, other):
        return not (self < other)

    def dot(self, other):
        if not isinstance(other, Vector):
            raise TypeError("argument must be a Vector")
        if self.dimension != other.dimension:
            raise ValueError("dimensions must match")
        return float(sum(a * b for a, b in zip(self.elements, other.elements)))

    def __mul__(self, other):
        if isinstance(other, (int, float)) and not isinstance(other, bool):
            return Vector(self.dimension, tuple(x * other for x in self.elements))
        if isinstance(other, Vector):
            if self.dimension != other.dimension:
                raise ValueError("dimensions must match")
            return Vector(self.dimension,
                          tuple(a * b for a, b in zip(self.elements, other.elements)))
        return NotImplemented

    def __rmul__(self, other):
        return self.__mul__(other)

    def __repr__(self):
        return f"Vector({self.dimension}, {self.elements})" 

### 初始化与默认值测试

In [12]:
v = Vector(3, (1, 2, 3))
zero = Vector(3)
print(v)
print(zero)
try:
    Vector(-2)
except ValueError as e:
    print("ValueError:", e)
try:
    Vector(2, (1, 2, 3))
except ValueError as e:
    print("ValueError:", e)

Vector(3, (1, 2, 3))
Vector(3, (0.0, 0.0, 0.0))
ValueError: dimension must be non-negative
ValueError: number of elements must match dimension


### 比较运算符测试（字典序，维度不同抛异常）

In [13]:
a = Vector(2, (2, 4))
b = Vector(2, (2, 5))
c = Vector(2, (2, 4))
print("a == b:", a == b, "| a == c:", a == c)
print("a < b:", a < b, "| a <= c:", a <= c, "| b > a:", b > a)
try:
    a < Vector(3, (1, 2, 3))
except ValueError as e:
    print("ValueError:", e)

a == b: False | a == c: True
a < b: True | a <= c: True | b > a: True
ValueError: cannot compare vectors of different dimensions


### `dot` 与乘法测试（标量左右乘、向量逐元素乘）

In [14]:
v = Vector(2, (2, 3))
w = Vector(2, (4, 5))
print("v.dot(w):", v.dot(w))
print("2 * v:", 2 * v)
print("v * 2:", v * 2)
print("v * w:", v * w)
try:
    v * Vector(3, (1, 2, 3))
except ValueError as e:
    print("ValueError:", e)

v.dot(w): 23.0
2 * v: Vector(2, (4, 6))
v * 2: Vector(2, (4, 6))
v * w: Vector(2, (8, 15))
ValueError: dimensions must match


### 2. 维度和元素应是类属性还是实例属性？

应使用**实例属性**。理由：每个向量的维度和元素取值各不相同，实例属性由每个对象各自持有、互不影响；若定义为类属性，则会被该类的所有实例共享，一个向量修改元素会影响其他向量，无法表示不同的向量。